#### RAG Pipeline
   
* Read Raw File ───▶ Document ───▶ Tokens ───▶ Embeddings ───▶ Vector Database ───▶ Ask User Query (Q) ───▶ Retrieve Top-k Docs ───▶ Context Assembly ───▶ LLM(Generator) ───▶ Final Answer

In [1]:
# !pip install langchain==0.3.27
# !pip install langchain-core==0.3.72
# !pip install langchain-community==0.3.27
# Suuportive packages for Langchain to read PDF Files.
!pip install pypdf
# Supportive packages for Langchain to read DOCX, TXTFiles.
!pip install tiktoken
# Install ChromaDB and Faiss for vector storage and similarity search
!pip install chromadb
!pip install faiss-cpu
!pip install google-genai


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma, FAISS
from langchain.chains import RetrievalQA

1. Load Text File Using TextLoader

In [3]:
text_loader = TextLoader("state_of_union.txt")

In [4]:
text_documents = text_loader.load()

In [5]:
print(text_documents[0])

page_content='Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russiaâ€™s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world. 

Groups of citizens blocking tanks with the

2. Load PDF File Using PyPDFLoader

In [6]:
pdf_loader = PyPDFLoader("D:\\OneDrive - Coforge Limited\\Desktop\\AI Training\\Resume\\1.pdf")

In [7]:
pdf_documents = pdf_loader.load()

In [8]:
# Length of the documents loaded from the PDF file
print(f"Number of documents loaded from PDF: {len(pdf_documents)}")

Number of documents loaded from PDF: 1


In [9]:
# Display the first document loaded from the PDF file
print(pdf_documents[0])
 

page_content='Functional Resume Sample 
 
John W. Smith  
2002 Front Range Way Fort Collins, CO 80525  
jwsmith@colostate.edu 
 
Career Summary 
 
Four years experience in early childhood development with a diverse background in the care of 
special needs children and adults.  
  
Adult Care Experience 
 
• Determined work placement for 150 special needs adult clients.  
• Maintained client databases and records.  
• Coordinated client contact with local health care professionals on a monthly basis.     
• Managed 25 volunteer workers.     
 
Childcare Experience 
 
• Coordinated service assignments for 20 part-time counselors and 100 client families. 
• Oversaw daily activity and outing planning for 100 clients.  
• Assisted families of special needs clients with researching financial assistance and 
healthcare. 
• Assisted teachers with managing daily classroom activities.    
• Oversaw daily and special student activities.     
 
Employment History  
 
1999-2002 Counseling Superviso

In [10]:
pdf_documents

[Document(metadata={'producer': 'Acrobat Distiller 5.0.5 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2004-10-20T11:36:31-06:00', 'moddate': '2004-10-20T11:36:31-06:00', 'author': 'Luann Barnes', 'title': 'Microsoft Word - Document6', 'source': 'D:\\OneDrive - Coforge Limited\\Desktop\\AI Training\\Resume\\1.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Functional Resume Sample \n \nJohn W. Smith  \n2002 Front Range Way Fort Collins, CO 80525  \njwsmith@colostate.edu \n \nCareer Summary \n \nFour years experience in early childhood development with a diverse background in the care of \nspecial needs children and adults.  \n  \nAdult Care Experience \n \n• Determined work placement for 150 special needs adult clients.  \n• Maintained client databases and records.  \n• Coordinated client contact with local health care professionals on a monthly basis.     \n• Managed 25 volunteer workers.     \n \nChildcare Experience \n \n• Coordinated ser

3. Read Multiple Files DirectoryLoader

In [11]:
!pip install unstructured
!pip install "unstructured[pdf]"
!pip install "unstructured[local-inference]"


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
loader = DirectoryLoader("./Resume", glob="**/*.pdf", loader_cls=PyPDFLoader)

In [13]:
documents = loader.load()

In [14]:
# display the first document loaded from the directory
print(documents[0].page_content)

Functional Resume Sample 
 
John W. Smith  
2002 Front Range Way Fort Collins, CO 80525  
jwsmith@colostate.edu 
 
Career Summary 
 
Four years experience in early childhood development with a diverse background in the care of 
special needs children and adults.  
  
Adult Care Experience 
 
• Determined work placement for 150 special needs adult clients.  
• Maintained client databases and records.  
• Coordinated client contact with local health care professionals on a monthly basis.     
• Managed 25 volunteer workers.     
 
Childcare Experience 
 
• Coordinated service assignments for 20 part-time counselors and 100 client families. 
• Oversaw daily activity and outing planning for 100 clients.  
• Assisted families of special needs clients with researching financial assistance and 
healthcare. 
• Assisted teachers with managing daily classroom activities.    
• Oversaw daily and special student activities.     
 
Employment History  
 
1999-2002 Counseling Supervisor, The Wesley 

In [15]:
print(f"Number of documents loaded from directory : {len(documents)}")

Number of documents loaded from directory : 14


4. Character Text Splitter -- Splitting the documents into smaller chunks(not tokens)



In [16]:
char_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", ""])

In [17]:
char_chunks = char_splitter.split_documents(documents)
 

In [18]:
char_chunks[0]

Document(metadata={'producer': 'Acrobat Distiller 5.0.5 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2004-10-20T11:36:31-06:00', 'moddate': '2004-10-20T11:36:31-06:00', 'author': 'Luann Barnes', 'title': 'Microsoft Word - Document6', 'source': 'Resume\\1.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Functional Resume Sample \n \nJohn W. Smith  \n2002 Front Range Way Fort Collins, CO 80525  \njwsmith@colostate.edu \n \nCareer Summary \n \nFour years experience in early childhood development with a diverse background in the care of \nspecial needs children and adults.  \n  \nAdult Care Experience')

In [19]:
len(char_chunks)

105